## Copying and patching votenet while compile pointnet2 

In [2]:
# ============================================
# Cell A — Install deps + patch + compile pointnet2 + load libtorch
# ============================================
import os, sys, shutil, subprocess, importlib
from pathlib import Path

# === Import torch FIRST so libc10.so / libtorch.so get loaded into the process ===
# (Otherwise pointnet2._ext won't find them at import time)
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}')

# === Install VoteNet's Python deps ===
print('\nInstalling Python deps...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'plyfile', 'trimesh', 'networkx',
                'opencv-python', 'matplotlib'],
               check=False)
print('  done.')

SRC_REPO  = Path('/kaggle/input/datasets/vathsal05/votenet-source/votenet_reference')
WORK_REPO = Path('/kaggle/working/votenet')

# === Copy source to writable location ===
if WORK_REPO.exists():
    shutil.rmtree(WORK_REPO)
print(f'\nCopying {SRC_REPO} -> {WORK_REPO}')
shutil.copytree(SRC_REPO, WORK_REPO)

# === Strip macOS AppleDouble files (._*) ===
n_removed = 0
for f in WORK_REPO.rglob('._*'):
    if f.is_file():
        f.unlink()
        n_removed += 1
print(f'  removed {n_removed} ._* AppleDouble files')

# === FIX 1: AT_CHECK -> TORCH_CHECK ===
print('\nPatching AT_CHECK -> TORCH_CHECK...')
ext_src = WORK_REPO / 'pointnet2' / '_ext_src'
n_patched = 0
for f in list(ext_src.rglob('*.cpp')) + list(ext_src.rglob('*.cu')) + list(ext_src.rglob('*.h')):
    if f.name.startswith('._'):
        continue
    txt = f.read_text()
    if 'AT_CHECK' in txt:
        f.write_text(txt.replace('AT_CHECK', 'TORCH_CHECK'))
        n_patched += 1
print(f'  patched {n_patched} files')

# === FIX 2: setup.py with absolute paths ===
print('\nRewriting setup.py with absolute paths...')
(WORK_REPO / 'pointnet2' / 'setup.py').write_text('''import os, glob
from setuptools import setup
from torch.utils.cpp_extension import BuildExtension, CUDAExtension
_ext_src_root = os.path.abspath("_ext_src")
_ext_sources = glob.glob(f"{_ext_src_root}/src/*.cpp") + glob.glob(f"{_ext_src_root}/src/*.cu")
_ext_sources = [s for s in _ext_sources if not os.path.basename(s).startswith("._")]
setup(
    name="pointnet2",
    ext_modules=[CUDAExtension(
        name="pointnet2._ext",
        sources=_ext_sources,
        extra_compile_args={
            "cxx":  ["-O2", f"-I{_ext_src_root}/include"],
            "nvcc": ["-O2", f"-I{_ext_src_root}/include"],
        },
    )],
    cmdclass={"build_ext": BuildExtension},
)
''')
print('  written.')

# === Compile, then manually copy .so to pointnet2/ ===
print('\nCompiling pointnet2 (~5 min)...')
os.chdir(WORK_REPO / 'pointnet2')

# Clean stale build cache
for sub in ['build', 'dist', 'pointnet2.egg-info']:
    if Path(sub).exists():
        shutil.rmtree(sub)

result = subprocess.run(
    [sys.executable, 'setup.py', 'build_ext'],
    capture_output=True, text=True
)
print('\n'.join((result.stdout + result.stderr).splitlines()[-10:]))
assert result.returncode == 0, f'pointnet2 build FAILED (exit {result.returncode})'

# Find the built .so anywhere under build/ and copy it to pointnet2/
built_sos = list(Path('build').rglob('_ext*.so'))
assert built_sos, 'No _ext .so file found under build/ after compile!'
target_dir = WORK_REPO / 'pointnet2'
for so in built_sos:
    dest = target_dir / so.name
    shutil.copy(so, dest)
    print(f'  copied {so} -> {dest}')

so_files = list(target_dir.glob('_ext*.so'))
print(f'\n_ext .so files in pointnet2/: {[f.name for f in so_files]}')
assert so_files
print('  build SUCCESS.')

# === Verify Python deps ===
print('\nVerifying Python deps...')
for mod in ['plyfile', 'trimesh', 'networkx', 'cv2', 'matplotlib']:
    try:
        importlib.import_module(mod)
        print(f'  OK   {mod}')
    except ImportError as e:
        print(f'  FAIL {mod}: {e}')

# === Clear stale cached imports (preserve torch!) ===
for k in list(sys.modules.keys()):
    if any(s in k for s in ['pointnet2', 'votenet', 'backbone_module',
                            'voting_module', 'proposal_module', 'loss_helper',
                            'dump_helper', 'model_util_sunrgbd', 'pc_util']):
        del sys.modules[k]
importlib.invalidate_caches()

# === sys.path setup ===
for p in [str(WORK_REPO), str(WORK_REPO / 'utils'),
          str(WORK_REPO / 'models'), str(WORK_REPO / 'sunrgbd')]:
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

# === Verify all imports (torch already loaded at top, so libc10 is in process) ===
import torch  # ensure it stays loaded
assert 'torch' in sys.modules, 'torch must be imported before pointnet2._ext'

import pointnet2._ext
print(f'\npointnet2._ext loaded from: {pointnet2._ext.__file__}')

from votenet import VoteNet
from loss_helper import get_loss
from model_util_sunrgbd import SunrgbdDatasetConfig
import pc_util
import sunrgbd_utils
print('\nAll imports OK. VoteNet repo ready at:', WORK_REPO)

torch=2.10.0+cu128  cuda=True

Installing Python deps...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 36.0 MB/s eta 0:00:00
  done.

Copying /kaggle/input/datasets/vathsal05/votenet-source/votenet_reference -> /kaggle/working/votenet
  removed 56 ._* AppleDouble files

Patching AT_CHECK -> TORCH_CHECK...
  patched 5 files

Rewriting setup.py with absolute paths...
  written.

Compiling pointnet2 (~5 min)...
                 from /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include/torch/all.h:8,
                 from /usr/local/lib/python3.12/dist-packages/torch/include/torch/extension.h:6,
                 from /kaggle/working/votenet/pointnet2/_ext_src/include/sampling.h:7,
                 from /kaggle/working/votenet/pointnet2/_ext_src/src/sampling.cpp:6:
/usr/local/lib/python3.12/dist-packages/torch/include/ATen/core/TensorBody.h:251:7: note: declared here
  251 | 

## 27-class configuration and computing mean sizes from data

In [3]:
# ============================================
# Cell B — 27-class config + mean sizes from training data
# ============================================
import numpy as np
from pathlib import Path
from model_util_sunrgbd import SunrgbdDatasetConfig

CLASS_NAMES = [
    'bed', 'table', 'sofa', 'chair', 'toilet',
    'desk', 'dresser', 'night_stand', 'bookshelf', 'bathtub',
    'ammo_box', 'binoculars', 'combat_knife', 'flashlight', 'gas_mask',
    'hand_grenade', 'helmet', 'magazine', 'military_radio', 'pistol',
    'rifle', 'rocket_launcher', 'shotgun', 'sniper_rifle',
    'tactical_backpack', 'tactical_vest', 'wire_cutter',
]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

DATA_ROOT = Path('/kaggle/input/datasets/vathsal05/votenet-synthetic-27class/synthetic_v1')
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR   = DATA_ROOT / 'val'

# Compute per-class mean (l, w, h) from training bboxes.
# Our stored bbox: (cx, cy, cz, X_ext, Y_ext, Z_ext, yaw, class) in Y-up world.
# After Y<->Z swap to convert to Z-up (which VoteNet expects):
#   new (l, w, h) = (X_ext, Z_ext, Y_ext)   [Y_ext becomes height in Z-up]
print(f'Scanning training bboxes for per-class mean sizes...')
sums = np.zeros((NUM_CLASSES, 3))
counts = np.zeros(NUM_CLASSES, dtype=int)
bbox_files = sorted([f for f in TRAIN_DIR.glob('*_bbox.npy') if not f.name.startswith('._')])
print(f'  {len(bbox_files)} bbox files')
for f in bbox_files:
    bb = np.load(f)
    for box in bb:
        cls = int(box[7])
        # convert Y-up extents to Z-up (l=X_ext, w=Z_ext, h=Y_ext)
        sums[cls] += [box[3], box[5], box[4]]
        counts[cls] += 1

mean_size_arr = sums / np.maximum(counts[:, None], 1)
print('\nMean sizes per class (l, w, h) in meters:')
for i, c in enumerate(CLASS_NAMES):
    print(f'  {c:18s}  l={mean_size_arr[i,0]:.3f}  w={mean_size_arr[i,1]:.3f}  '
          f'h={mean_size_arr[i,2]:.3f}  ({counts[i]} instances)')


class SyntheticDatasetConfig:
    """VoteNet config for our 27-class synthetic dataset (mimics SunrgbdDatasetConfig)."""
    def __init__(self, mean_size_arr):
        self.num_class = NUM_CLASSES
        self.num_heading_bin = 12
        self.num_size_cluster = NUM_CLASSES

        self.type2class    = {c: i for i, c in enumerate(CLASS_NAMES)}
        self.class2type    = {i: c for i, c in enumerate(CLASS_NAMES)}
        self.type2onehotclass = dict(self.type2class)

        self.mean_size_arr = mean_size_arr.astype(np.float32)
        self.type_mean_size = {c: self.mean_size_arr[i] for i, c in enumerate(CLASS_NAMES)}

    def size2class(self, size, type_name):
        return self.type2class[type_name], size - self.type_mean_size[type_name]

    def class2size(self, pred_cls, residual):
        return self.type_mean_size[self.class2type[pred_cls]] + residual

    def angle2class(self, angle):
        num = self.num_heading_bin
        angle = angle % (2*np.pi)
        per = 2*np.pi / num
        shifted = (angle + per/2) % (2*np.pi)
        cls = int(shifted / per)
        residual = shifted - (cls*per + per/2)
        return cls, residual

    def class2angle(self, pred_cls, residual, to_label_format=True):
        per = 2*np.pi / self.num_heading_bin
        angle = pred_cls * per + residual
        if to_label_format and angle > np.pi:
            angle -= 2*np.pi
        return angle

    def param2obb(self, center, hc, hr, sc, sr):
        obb = np.zeros((7,))
        obb[0:3] = center
        obb[3:6] = self.class2size(int(sc), sr)
        obb[6] = -self.class2angle(hc, hr)
        return obb


DC = SyntheticDatasetConfig(mean_size_arr)
print(f'\nConfig built: {DC.num_class} classes, {DC.num_heading_bin} heading bins, '
      f'{DC.num_size_cluster} size clusters')
print(f'mean_size_arr.shape = {DC.mean_size_arr.shape}')

Scanning training bboxes for per-class mean sizes...
  5000 bbox files

Mean sizes per class (l, w, h) in meters:
  bed                 l=2.159  w=2.173  h=0.816  (779 instances)
  table               l=1.464  w=1.464  h=0.608  (1279 instances)
  sofa                l=1.744  w=1.765  h=0.686  (1140 instances)
  chair               l=0.462  w=0.463  h=0.654  (1616 instances)
  toilet              l=0.519  w=0.518  h=0.635  (1574 instances)
  desk                l=1.298  w=1.311  h=0.909  (1372 instances)
  dresser             l=0.751  w=0.755  h=1.067  (1647 instances)
  night_stand         l=0.547  w=0.547  h=0.550  (1577 instances)
  bookshelf           l=0.579  w=0.576  h=1.084  (1573 instances)
  bathtub             l=1.456  w=1.464  h=0.644  (1268 instances)
  ammo_box            l=0.329  w=0.329  h=0.182  (1604 instances)
  binoculars          l=0.210  w=0.209  h=0.082  (1636 instances)
  combat_knife        l=0.206  w=0.205  h=0.069  (1673 instances)
  flashlight          l=0.051

## Synthetic dataset class (Y-up → Z-up conversion)

In [4]:
# ============================================
# Cell C — SyntheticVoteNetDataset
# ============================================
import torch
from torch.utils.data import Dataset
import pc_util  # from votenet utils
import sunrgbd_utils

MAX_NUM_OBJ = 64


def _read_npz_first(path):
    """Read first array from .npz regardless of key name (handles 'pc'/'point_votes')."""
    with np.load(path) as f:
        return f[f.files[0]]


class SyntheticVoteNetDataset(Dataset):
    """
    Reads our synthetic .npz/.npy files. Converts Y-up -> Z-up to match VoteNet conventions.
    Returns dict compatible with VoteNet's loss + APCalculator.
    """
    def __init__(self, data_dir, num_points=20000, augment=False, dataset_config=None):
        self.data_dir = Path(data_dir)
        self.num_points = num_points
        self.augment = augment
        self.DC = dataset_config

        # Discover scene IDs (filter macOS resource forks)
        pc_files = sorted([f for f in self.data_dir.glob('*_pc.npz')
                           if not f.name.startswith('._')])
        self.scan_names = [f.name.replace('_pc.npz', '') for f in pc_files]
        print(f'  {self.data_dir.name}: {len(self.scan_names)} scenes')

    def __len__(self):
        return len(self.scan_names)

    def __getitem__(self, idx):
        sid = self.scan_names[idx]
        pc        = _read_npz_first(self.data_dir / f'{sid}_pc.npz').astype(np.float32)      # (N, 6)
        bboxes    = np.load(self.data_dir / f'{sid}_bbox.npy').astype(np.float32)            # (K, 8)
        votes_raw = _read_npz_first(self.data_dir / f'{sid}_votes.npz').astype(np.float32)   # (N, 10)

        # === Convert Y-up -> Z-up: swap axes 1 and 2 ===
        # pc: (X, Y_up, Z, R, G, B) -> (X, Z, Y_up, R, G, B)
        pc = pc[:, [0, 2, 1, 3, 4, 5]]
        # bboxes: (cx, cy_up, cz, X_ext, Y_ext, Z_ext, yaw, class) ->
        #          (cx, cz, cy_up, X_ext, Z_ext, Y_ext, yaw, class)
        bboxes = bboxes[:, [0, 2, 1, 3, 5, 4, 6, 7]]
        # votes_raw: (mask, v1x, v1y, v1z, v2x, v2y, v2z, v3x, v3y, v3z) ->
        #            (mask, v1x, v1z, v1y, v2x, v2z, v2y, v3x, v3z, v3y)
        votes_raw = votes_raw[:, [0, 1, 3, 2, 4, 6, 5, 7, 9, 8]]

        # === Drop RGB (all zeros), keep XYZ + height feature ===
        point_cloud = pc[:, 0:3]
        # use_height: append (Z - floor_height) as feature
        floor_height = np.percentile(point_cloud[:, 2], 0.99)
        height = point_cloud[:, 2] - floor_height
        point_cloud = np.concatenate([point_cloud, height[:, None]], axis=1)  # (N, 4)

        # === Augmentation (Z-up world) ===
        if self.augment:
            # 50% flip along YZ plane (X -> -X)
            if np.random.random() > 0.5:
                point_cloud[:, 0] *= -1
                bboxes[:, 0] *= -1
                bboxes[:, 6] = np.pi - bboxes[:, 6]
                votes_raw[:, [1, 4, 7]] *= -1

            # Rotation around Z by [-30, +30] deg
            rot_angle = (np.random.random() - 0.5) * (np.pi / 3)
            rot_mat = sunrgbd_utils.rotz(rot_angle)

            votes_end = np.zeros_like(votes_raw)
            for k, start in enumerate([1, 4, 7]):
                votes_end[:, start:start+3] = (point_cloud[:, 0:3] +
                                               votes_raw[:, start:start+3]) @ rot_mat.T
            point_cloud[:, 0:3] = point_cloud[:, 0:3] @ rot_mat.T
            bboxes[:, 0:3] = bboxes[:, 0:3] @ rot_mat.T
            bboxes[:, 6] -= rot_angle
            for start in [1, 4, 7]:
                votes_raw[:, start:start+3] = votes_end[:, start:start+3] - point_cloud[:, 0:3]

            # Scale 0.85-1.15
            s = np.random.random() * 0.3 + 0.85
            point_cloud[:, 0:3] *= s
            point_cloud[:, -1] *= s
            bboxes[:, 0:6] *= s
            for start in [1, 4, 7]:
                votes_raw[:, start:start+3] *= s

        # === Build labels (MAX_NUM_OBJ-padded arrays) ===
        n_box = len(bboxes)
        center_label   = np.zeros((MAX_NUM_OBJ, 3), dtype=np.float32)
        heading_class  = np.zeros((MAX_NUM_OBJ,), dtype=np.int64)
        heading_resid  = np.zeros((MAX_NUM_OBJ,), dtype=np.float32)
        size_class     = np.zeros((MAX_NUM_OBJ,), dtype=np.int64)
        size_resid     = np.zeros((MAX_NUM_OBJ, 3), dtype=np.float32)
        sem_cls        = np.zeros((MAX_NUM_OBJ,), dtype=np.int64)
        label_mask     = np.zeros((MAX_NUM_OBJ,), dtype=np.float32)
        label_mask[:n_box] = 1

        for i in range(n_box):
            box = bboxes[i]
            cls_id = int(box[7])
            cls_name = CLASS_NAMES[cls_id]
            box_size = box[3:6]  # full extents (already correct, no *2 needed)
            ac, ar = self.DC.angle2class(box[6])
            sc, sr = self.DC.size2class(box_size, cls_name)
            heading_class[i] = ac
            heading_resid[i] = ar
            size_class[i] = sc
            size_resid[i] = sr
            sem_cls[i] = cls_id

        # === Use the axis-aligned center from corners (matches SUN RGB-D loader) ===
        target_center = np.zeros((MAX_NUM_OBJ, 3), dtype=np.float32)
        for i in range(n_box):
            box = bboxes[i]
            corners = sunrgbd_utils.my_compute_box_3d(box[0:3], box[3:6], box[6])
            target_center[i] = corners.mean(axis=0)

        # === Subsample point cloud + votes to num_points ===
        point_cloud, choices = pc_util.random_sampling(point_cloud, self.num_points, return_choices=True)
        vote_mask = votes_raw[choices, 0].astype(np.int64)
        vote_label = votes_raw[choices, 1:].astype(np.float32)

        return {
            'point_clouds':            point_cloud.astype(np.float32),
            'center_label':            target_center,
            'heading_class_label':     heading_class,
            'heading_residual_label':  heading_resid,
            'size_class_label':        size_class,
            'size_residual_label':     size_resid,
            'sem_cls_label':           sem_cls,
            'box_label_mask':          label_mask,
            'vote_label':              vote_label,
            'vote_label_mask':         vote_mask,
            'scan_idx':                np.array(idx).astype(np.int64),
            'max_gt_bboxes':           np.zeros((MAX_NUM_OBJ, 8), dtype=np.float32),
        }


# Quick sanity-check
print('Building datasets...')
train_ds = SyntheticVoteNetDataset(TRAIN_DIR, num_points=20000, augment=True, dataset_config=DC)
val_ds   = SyntheticVoteNetDataset(VAL_DIR,   num_points=20000, augment=False, dataset_config=DC)

sample = train_ds[0]
print(f'\nSample keys: {sorted(sample.keys())}')
for k, v in sample.items():
    print(f'  {k:25s} {v.shape if hasattr(v, "shape") else v}  dtype={v.dtype if hasattr(v, "dtype") else type(v).__name__}')

assert sample['point_clouds'].shape == (20000, 4), 'pc must be (N, 4) with use_height'
assert sample['box_label_mask'].sum() > 0, 'scene must have at least 1 object'
print(f'\nDataset OK. Train={len(train_ds)} scenes, Val={len(val_ds)} scenes')

Building datasets...
  train: 5000 scenes
  val: 1000 scenes

Sample keys: ['box_label_mask', 'center_label', 'heading_class_label', 'heading_residual_label', 'max_gt_bboxes', 'point_clouds', 'scan_idx', 'sem_cls_label', 'size_class_label', 'size_residual_label', 'vote_label', 'vote_label_mask']
  point_clouds              (20000, 4)  dtype=float32
  center_label              (64, 3)  dtype=float32
  heading_class_label       (64,)  dtype=int64
  heading_residual_label    (64,)  dtype=float32
  size_class_label          (64,)  dtype=int64
  size_residual_label       (64, 3)  dtype=float32
  sem_cls_label             (64,)  dtype=int64
  box_label_mask            (64,)  dtype=float32
  vote_label                (20000, 9)  dtype=float32
  vote_label_mask           (20000,)  dtype=int64
  scan_idx                  ()  dtype=int64
  max_gt_bboxes             (64, 8)  dtype=float32

Dataset OK. Train=5000 scenes, Val=1000 scenes


## Building model and transfering Pre-trained weights and doing a loss sanity check

In [5]:
# ============================================
# Cell D — Build VoteNet for 27 classes + transfer Phase 7 weights
# ============================================
import torch
from torch.utils.data import DataLoader
from votenet import VoteNet
from loss_helper import get_loss

device = torch.device('cuda')

model = VoteNet(
    num_class=DC.num_class,
    num_heading_bin=DC.num_heading_bin,
    num_size_cluster=DC.num_size_cluster,
    mean_size_arr=DC.mean_size_arr,
    num_proposal=256,
    input_feature_dim=1,    # height feature only
    vote_factor=1,
    sampling='vote_fps',
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'VoteNet built. Params: {n_params/1e6:.2f}M')

# === Selective weight transfer from Phase 7 ===
CKPT = '/kaggle/input/datasets/vathsal05/votenet-sunrgbd-trained/votenet_sunrgbd_best.pt'
ckpt = torch.load(CKPT, map_location=device, weights_only=False)
src_state = ckpt.get('model_state_dict', ckpt.get('model', ckpt))
if not isinstance(src_state, dict) or 'backbone_net' not in str(list(src_state.keys())[0]):
    # If not a state_dict format, find the right key
    print(f'Checkpoint top-level keys: {list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt)}')

tgt_state = dict(model.state_dict())
loaded, skipped_shape, skipped_missing = [], [], []
for k, v in tgt_state.items():
    if k in src_state and src_state[k].shape == v.shape:
        tgt_state[k] = src_state[k].clone()
        loaded.append(k)
    elif k in src_state:
        skipped_shape.append((k, src_state[k].shape, v.shape))
    else:
        skipped_missing.append(k)
model.load_state_dict(tgt_state)

print(f'\nWeight transfer:')
print(f'  loaded:  {len(loaded)} layers')
print(f'  shape mismatch (re-init): {len(skipped_shape)} layers')
print(f'  not in checkpoint:        {len(skipped_missing)} layers')
print(f'\nShape-mismatched layers (expected — these had 10 classes, now 27):')
for k, src_sh, tgt_sh in skipped_shape[:10]:
    print(f'  {k}:  src={src_sh}  tgt={tgt_sh}')

# === DataLoader + loss sanity check ===
BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True, drop_last=False)
print(f'\nDataLoaders: train={len(train_loader)} batches, val={len(val_loader)} batches')

LABEL_KEYS = ['center_label', 'heading_class_label', 'heading_residual_label',
              'size_class_label', 'size_residual_label', 'sem_cls_label',
              'box_label_mask', 'vote_label', 'vote_label_mask']

print('\nSanity-checking loss on one batch...')
batch = next(iter(train_loader))
batch_gpu = {k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
             for k, v in batch.items()}
model.train()
end_points = model(batch_gpu)
for k in LABEL_KEYS:
    end_points[k] = batch_gpu[k]
loss, end_points = get_loss(end_points, DC)
print(f'  loss = {loss.item():.4f}  (expected 25-50 for untrained final layers)')
assert not torch.isnan(loss), 'Loss is NaN — pipeline broken!'
print('  pipeline OK.')

VoteNet built. Params: 0.96M

Weight transfer:
  loaded:  144 layers
  shape mismatch (re-init): 2 layers
  not in checkpoint:        0 layers

Shape-mismatched layers (expected — these had 10 classes, now 27):
  pnet.conv3.weight:  src=torch.Size([79, 128, 1])  tgt=torch.Size([164, 128, 1])
  pnet.conv3.bias:  src=torch.Size([79])  tgt=torch.Size([164])

DataLoaders: train=625 batches, val=125 batches

Sanity-checking loss on one batch...
  loss = 19.5591  (expected 25-50 for untrained final layers)
  pipeline OK.


/kaggle/working/votenet/models/loss_helper.py:155: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  heading_label_one_hot = torch.cuda.FloatTensor(batch_size, heading_class_label.shape[1], num_heading_bin).zero_()


## Training loop (30 epochs)

In [6]:
# ============================================
# Cell E — Training loop
# ============================================
import time, json
from collections import defaultdict
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

EPOCHS  = 30
LR_INIT = 1e-4      # lower than Phase 7 (1e-3) since we're fine-tuning
LR_MIN  = 1e-6

optimizer = AdamW(model.parameters(), lr=LR_INIT, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

best_val_loss = float('inf')
history = []


def run_epoch(loader, training):
    model.train() if training else model.eval()
    sums = defaultdict(float)
    n = 0
    for batch in loader:
        batch_gpu = {k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
                     for k, v in batch.items()}
        if training:
            optimizer.zero_grad(set_to_none=True)
        ctx = torch.enable_grad() if training else torch.no_grad()
        with ctx:
            end_points = model(batch_gpu)
            for k in LABEL_KEYS:
                end_points[k] = batch_gpu[k]
            loss, end_points = get_loss(end_points, DC)
        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        sums['loss'] += loss.item()
        for k in ['vote_loss', 'objectness_loss', 'center_loss',
                  'heading_cls_loss', 'heading_reg_loss',
                  'size_cls_loss', 'size_reg_loss', 'sem_cls_loss', 'box_loss']:
            if k in end_points and isinstance(end_points[k], torch.Tensor):
                sums[k] += end_points[k].item()
        n += 1
    return {k: v / n for k, v in sums.items()}


for ep in range(EPOCHS):
    t0 = time.time()
    train_m = run_epoch(train_loader, training=True)
    cur_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    val_m = {}
    if (ep + 1) % 2 == 0 or ep == EPOCHS - 1:
        val_m = run_epoch(val_loader, training=False)

    val_str = f'  val_loss={val_m.get("loss", float("nan")):.3f}' if val_m else ''
    print(f'Epoch {ep+1:3d}/{EPOCHS}  lr={cur_lr:.6f}  '
          f'train_loss={train_m["loss"]:.3f}{val_str}  ({time.time()-t0:.0f}s)',
          flush=True)

    history.append({'epoch': ep+1, 'lr': cur_lr,
                    **{f'train_{k}': v for k, v in train_m.items()},
                    **{f'val_{k}':   v for k, v in val_m.items()}})

    # Save best by val loss
    if val_m and val_m['loss'] < best_val_loss:
        best_val_loss = val_m['loss']
        torch.save({'epoch': ep+1, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_loss': val_m['loss']},
                   OUT_DIR / 'votenet_27class_best.pt')
        print(f'  -> saved best (val_loss={val_m["loss"]:.3f})')

    # Always save last
    torch.save({'epoch': ep+1, 'model_state_dict': model.state_dict()},
               OUT_DIR / 'votenet_27class_last.pt')

    with open(OUT_DIR / 'training_history_phase8.json', 'w') as f:
        json.dump(history, f, indent=2)

print(f'\nTraining complete. Best val_loss: {best_val_loss:.4f}')
print(f'Best checkpoint: {OUT_DIR/"votenet_27class_best.pt"}')

Epoch   1/30  lr=0.000100  train_loss=13.993  (247s)
Epoch   2/30  lr=0.000100  train_loss=10.141  val_loss=9.062  (284s)
  -> saved best (val_loss=9.062)
Epoch   3/30  lr=0.000099  train_loss=8.703  (260s)
Epoch   4/30  lr=0.000098  train_loss=7.973  val_loss=7.449  (283s)
  -> saved best (val_loss=7.449)
Epoch   5/30  lr=0.000096  train_loss=7.441  (260s)
Epoch   6/30  lr=0.000093  train_loss=7.028  val_loss=6.604  (284s)
  -> saved best (val_loss=6.604)
Epoch   7/30  lr=0.000091  train_loss=6.773  (260s)
Epoch   8/30  lr=0.000087  train_loss=6.520  val_loss=6.206  (283s)
  -> saved best (val_loss=6.206)
Epoch   9/30  lr=0.000084  train_loss=6.320  (260s)
Epoch  10/30  lr=0.000080  train_loss=6.160  val_loss=5.881  (283s)
  -> saved best (val_loss=5.881)
Epoch  11/30  lr=0.000075  train_loss=6.066  (260s)
Epoch  12/30  lr=0.000071  train_loss=5.916  val_loss=5.624  (283s)
  -> saved best (val_loss=5.624)
Epoch  13/30  lr=0.000066  train_loss=5.845  (260s)
Epoch  14/30  lr=0.000061  t

## Visualizations

In [5]:
# ============================================
# Cell: Demo gallery — one scene per category
# ============================================
import sys, importlib
sys.path.insert(0, '/Users/dosvatsky/3D Object Detection/scripts')
if 'interactive_viz_plotly_v2' in sys.modules:
    importlib.reload(sys.modules['interactive_viz_plotly_v2'])
from interactive_viz_plotly_v2 import show_scene

# Pick scenes the analyzer surfaced. Replace these numbers with whatever
# `find_demo_scenes.py` printed in Step 1 — they vary based on your run.
DEMO_GALLERY = {
    'BEST performance':       778,   # high precision × recall
    'DENSEST':                778,
    'MOST DIVERSE classes':   271,
    'WORST recall (honest)':  216,
    'FURNITURE-only':         408,
    'MILITARY-heavy':         802,
    'TINY-object case':       216,
}

for label, scan_idx in DEMO_GALLERY.items():
    print(f'\n--- {label}: Scene {scan_idx} ---')
    fig = show_scene(scan_idx, score_threshold=0.5)
    fig.update_layout(title_text=f'[{label}] ' + fig.layout.title.text,
                      height=700)
    fig.show()


--- BEST performance: Scene 778 ---



--- DENSEST: Scene 778 ---



--- MOST DIVERSE classes: Scene 271 ---



--- WORST recall (honest): Scene 216 ---



--- FURNITURE-only: Scene 408 ---



--- MILITARY-heavy: Scene 802 ---



--- TINY-object case: Scene 216 ---


In [9]:
# A varied tour — paste, run, scroll through during your demo
SCENES_TO_TOUR = [
    (334, 'furniture-heavy with tactical_backpack'),
    (802, 'sofa, ammo_box, binoculars — high confidence'),
    (216, 'tiny objects (flashlight, hand_grenade, combat_knife)'),
    (50,  'unseen variety'),
    
    
]

for scan_idx, description in SCENES_TO_TOUR:
    print(f'\n=== Scene {scan_idx} — {description} ===')
    show_scene(scan_idx, score_threshold=0.5, top_k=15).show()


=== Scene 334 — furniture-heavy with tactical_backpack ===



=== Scene 802 — sofa, ammo_box, binoculars — high confidence ===



=== Scene 216 — tiny objects (flashlight, hand_grenade, combat_knife) ===



=== Scene 50 — unseen variety ===


In [7]:
# Same scene, two confidence levels — shows the precision-recall trade-off
scan_idx = 778

print(f'\n=== Scene {scan_idx} — relaxed threshold (score >= 0.3) ===')
show_scene(scan_idx, score_threshold=0.3, top_k=20).show()

print(f'\n=== Scene {scan_idx} — strict threshold (score >= 0.7) ===')
show_scene(scan_idx, score_threshold=0.7, top_k=20).show()


=== Scene 778 — relaxed threshold (score >= 0.3) ===



=== Scene 778 — strict threshold (score >= 0.7) ===
